# 🏛️ Regional Government Fiscal Performance Analysis
## Riau Islands Province, Indonesia (2018–2022)

**Author:** Umul Hasanah, S.M.  
**Data Source:** Official Budget Realization Reports (LRA) — BPKAD Riau Islands Province  
**Based on:** Undergraduate Thesis, UNRIKA 2024 (GPA 3.75, Cum Laude)  
**Tools:** Python · Pandas · Matplotlib

---

## 📦 STEP 1 — Load & Cek Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Load dataset real dari LRA Provinsi Kepulauan Riau
df = pd.read_csv('riau_islands_fiscal_data_REAL.csv')

print('=== DATA FISKAL PROVINSI KEPULAUAN RIAU 2018-2022 ===')
print(df[['year','decentralization_ratio_pct','independence_ratio_pct',
          'effectiveness_ratio_pct','efficiency_ratio_pct']].to_string(index=False))

print('\n=== CEK DATA KOSONG ===')
print(df.isnull().sum())

print(f'\nTotal data: {len(df)} tahun ({df["year"].min()}–{df["year"].max()})')

## 🧮 STEP 2 — Verifikasi Ulang Rasio dari Data Mentah

In [ ]:
# Hitung ulang semua rasio dari data mentah untuk verifikasi
df['decentralization_check'] = (df['local_own_revenue_pad'] / df['total_revenue'] * 100).round(2)
df['independence_check']     = (df['local_own_revenue_pad'] / df['central_transfers'] * 100).round(2)
df['effectiveness_check']    = (df['realized_pad'] / df['budget_pad'] * 100).round(2)
df['efficiency_check']       = (df['total_expenditure'] / df['total_revenue'] * 100).round(2)

print('=== VERIFIKASI PERHITUNGAN RASIO ===')
print('(Angka harus sama dengan kolom _ratio_pct di dataset)')
print()
for _, row in df.iterrows():
    print(f"Tahun {int(row['year'])}:")
    print(f"  Desentralisasi : {row['decentralization_check']}%  → {row['decentralization_criteria']}")
    print(f"  Kemandirian    : {row['independence_check']}%  → {row['independence_criteria']}")
    print(f"  Efektivitas PAD: {row['effectiveness_check']}%  → {row['effectiveness_criteria']}")
    print(f"  Efisiensi      : {row['efficiency_check']}%  → {row['efficiency_criteria']}")
    print()

## 📊 STEP 3 — Statistik Ringkas

In [ ]:
# Ubah PAD ke satuan miliar untuk lebih mudah dibaca
df['pad_billion'] = df['local_own_revenue_pad'] / 1_000_000_000
df['revenue_billion'] = df['total_revenue'] / 1_000_000_000
df['expenditure_billion'] = df['total_expenditure'] / 1_000_000_000

print('=== RINGKASAN KEUANGAN (IDR Miliar) ===')
print(f"{'Tahun':<8} {'PAD':>12} {'Total Pendapatan':>18} {'Total Belanja':>15}")
print('-' * 56)
for _, row in df.iterrows():
    print(f"{int(row['year']):<8} {row['pad_billion']:>11.1f}B  {row['revenue_billion']:>16.1f}B  {row['expenditure_billion']:>13.1f}B")

print(f"\n=== STATISTIK RASIO 2018-2022 ===")
print(f"Rata-rata Desentralisasi : {df['decentralization_ratio_pct'].mean():.2f}%  (Kriteria: Cukup)")
print(f"Rata-rata Kemandirian    : {df['independence_ratio_pct'].mean():.2f}%  (Kriteria: Sedang)")
print(f"Rata-rata Efektivitas PAD: {df['effectiveness_ratio_pct'].mean():.2f}% (Kriteria: Sangat Efektif)")
print(f"Rata-rata Efisiensi      : {df['efficiency_ratio_pct'].mean():.2f}% (Kriteria: Tidak Efisien)")

# Trend PAD
pad_growth = ((df['local_own_revenue_pad'].iloc[-1] / df['local_own_revenue_pad'].iloc[0]) - 1) * 100
print(f"\nPertumbuhan PAD 2018→2022: +{pad_growth:.1f}%")

## 📈 STEP 4 — Visualisasi

In [ ]:
# ── GRAFIK 1: Semua Rasio dalam Satu Grafik ──────────
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#1F4E79', '#2E86C1', '#27AE60', '#E74C3C']
ratios = [
    ('decentralization_ratio_pct', 'Fiscal Decentralization Ratio', colors[0]),
    ('independence_ratio_pct',     'Fiscal Independence Ratio',     colors[1]),
    ('effectiveness_ratio_pct',    'PAD Effectiveness Ratio',       colors[2]),
    ('efficiency_ratio_pct',       'Fiscal Efficiency Ratio',       colors[3]),
]

for col, label, color in ratios:
    ax.plot(df['year'], df[col], marker='o', label=label, color=color, linewidth=2.5, markersize=8)
    # Label angka di setiap titik
    for _, row in df.iterrows():
        ax.annotate(f"{row[col]:.1f}%",
                    (row['year'], row[col]),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=7.5, color=color)

# Garis referensi 100%
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='100% Reference Line')

ax.set_title('Fiscal Performance Ratios — Riau Islands Province (2018–2022)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Ratio (%)', fontsize=11)
ax.set_xticks(df['year'])
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(20, 130)
plt.tight_layout()
plt.savefig('all_ratios_trend.png', dpi=150)
plt.show()
print('✅ Grafik all_ratios_trend.png tersimpan!')

In [ ]:
# ── GRAFIK 2: PAD vs Total Pendapatan (Bar Chart) ────
fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(df))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], df['pad_billion'],
               width, label='PAD (Local Own Revenue)', color='#1F4E79')
bars2 = ax.bar([i + width/2 for i in x], df['revenue_billion'],
               width, label='Total Revenue', color='#85C1E9')

# Label angka
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{bar.get_height():.0f}B', ha='center', va='bottom', fontsize=8, color='#1F4E79')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{bar.get_height():.0f}B', ha='center', va='bottom', fontsize=8, color='#2E86C1')

ax.set_title('PAD vs Total Revenue — Riau Islands Province (IDR Billion)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Amount (IDR Billion)')
ax.set_xticks(x)
ax.set_xticklabels(df['year'])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('pad_vs_revenue.png', dpi=150)
plt.show()
print('✅ Grafik pad_vs_revenue.png tersimpan!')

In [ ]:
# ── GRAFIK 3: Desentralisasi dengan Zona Kriteria ────
fig, ax = plt.subplots(figsize=(12, 6))

# Zona warna kriteria
ax.axhspan(0, 10,    alpha=0.08, color='red',    label='Very Poor (0–10%)')
ax.axhspan(10, 20,   alpha=0.08, color='orange',  label='Poor (10–20%)')
ax.axhspan(20, 30,   alpha=0.08, color='yellow',  label='Moderate (20–30%)')
ax.axhspan(30, 40,   alpha=0.08, color='lightblue', label='Sufficient (30–40%)')
ax.axhspan(40, 50,   alpha=0.08, color='lightgreen', label='Good (40–50%)')
ax.axhspan(50, 100,  alpha=0.08, color='green',   label='Very Good (>50%)')

ax.plot(df['year'], df['decentralization_ratio_pct'],
        marker='o', color='#1F4E79', linewidth=3, markersize=10, zorder=5)

for _, row in df.iterrows():
    ax.annotate(f"{row['decentralization_ratio_pct']}%\n({row['decentralization_criteria'].split('/')[1].strip()})",
                (row['year'], row['decentralization_ratio_pct']),
                textcoords='offset points', xytext=(0, 15),
                ha='center', fontsize=9, color='#1F4E79', fontweight='bold')

ax.set_title('Fiscal Decentralization Ratio with Criteria Zones (2018–2022)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Decentralization Ratio (%)')
ax.set_xticks(df['year'])
ax.set_ylim(0, 60)
ax.legend(loc='upper left', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('decentralization_zones.png', dpi=150)
plt.show()
print('✅ Grafik decentralization_zones.png tersimpan!')

## 💡 STEP 5 — Insight & Kesimpulan

In [ ]:
print('=' * 60)
print('     KEY FINDINGS & POLICY RECOMMENDATIONS')
print('     Riau Islands Province Fiscal Analysis 2018-2022')
print('=' * 60)

print(f"""
📌 FINDING 1 — Fiscal Decentralization: SUFFICIENT → GOOD
   Ratio improved from 34.88% (2018) to 42.77% (2022).
   2018–2021: Sufficient category (30–40%)
   2022: Entered Good category (40–50%) — positive trend!
   PAD grew +{((df['local_own_revenue_pad'].iloc[-1]/df['local_own_revenue_pad'].iloc[0])-1)*100:.1f}% over 5 years.
   → Recommendation: Continue optimizing local tax collection
     and develop trade/manufacturing sectors.
""")

print(f"""📌 FINDING 2 — Fiscal Independence: MODERATE THROUGHOUT
   Ratio ranged 49.94%–74.77%, all in Moderate category.
   Best year: 2022 at 74.77% (close to reaching High category).
   Province still relies on central government transfers.
   → Recommendation: Increase community education on natural
     resource management to grow local revenue base.
""")

print(f"""📌 FINDING 3 — PAD Effectiveness: VERY EFFECTIVE
   All years exceeded 100% except 2021 (95.80%).
   Best year: 2022 at 116.00% — PAD exceeded budget target.
   2021 dip likely related to COVID-19 economic impacts.
   → Recommendation: Continue optimizing PAD realization
     and maintain 2022 momentum.
""")

print(f"""📌 FINDING 4 — Fiscal Efficiency: INEFFICIENT
   Most years showed efficiency ratio >100% (overspending).
   Only 2020 was efficient at 91.16% — likely due to
   COVID-19 budget cuts reducing expenditure.
   → Recommendation: Prepare regional officials by field
     expertise to reduce unnecessary regional expenditure.
""")

print('=' * 60)
print('Data Source: LRA BPKAD Riau Islands Province 2018-2022')
print('Analysis: Umul Hasanah, S.M. | UNRIKA 2024')
print('=' * 60)

---
## ✅ Project Complete!

**Files generated:**
- `all_ratios_trend.png` — semua rasio dalam satu grafik
- `pad_vs_revenue.png` — PAD vs total pendapatan
- `decentralization_zones.png` — desentralisasi dengan zona kriteria

**Data source:** Official LRA reports — BPKAD Riau Islands Province  
**Note:** This analysis is the Python-based reproduction of undergraduate thesis research (UNRIKA, 2024) 🎓